# `railroad` Fundamentals

In [ ]:
from railroad.core import (
    Fluent,
    State,
    Action,
    Effect,
    Operator,
    GroundedEffect,
    transition,
    get_next_actions,
    get_action_by_name,
)

F = Fluent

## Fluents and States

In [ ]:
f1 = F("at", "r1", "roomA")
f2 = F("at r1 roomA")
print(f1, "==", f2, "->", f1 == f2)

print(~f1)
print(F("not at r1 roomA") == ~f1)

In [ ]:
s = State(
    time=0.0,
    fluents={
        F("at r1 roomA"),
        F("free r1"),
    },
)
print(s)
print("time:", s.time)
print("fluents:", s.fluents)
print("upcoming_effects:", s.upcoming_effects)

## Actions and State Transitions

In [ ]:
move_r1 = Action(
    name="move r1 roomA roomB",
    preconditions={F("at r1 roomA"), F("free r1")},
    effects=[
        # at t=0: r1 is busy and no longer at roomA
        GroundedEffect(0.0, {~F("free r1"), ~F("at r1 roomA")}),
        # at t=5: r1 arrives at roomB and is free again
        GroundedEffect(5.0, {F("free r1"), F("at r1 roomB")}),
    ],
)
print(move_r1)

In [ ]:
print("precondition satisfied?", s.satisfies_precondition(move_r1))

In [ ]:
# transition() runs the world forward until at least one robot is free
outcomes = transition(s, move_r1)
print("number of outcomes:", len(outcomes))

s_next, prob = outcomes[0]
print("probability:", prob)
print("time:", s_next.time)
print("fluents:", s_next.fluents)
print("upcoming_effects:", s_next.upcoming_effects)

## Multi-Robot Concurrency

In [ ]:
# A lifted operator: ?r, ?from, ?to are parameters
move_op = Operator(
    name="move",
    parameters=[("?r", "robot"), ("?from", "location"), ("?to", "location")],
    preconditions=[F("at ?r ?from"), F("free ?r")],
    effects=[
        Effect(time=0, resulting_fluents={~F("free ?r"), ~F("at ?r ?from")}),
        Effect(time=5, resulting_fluents={F("free ?r"), F("at ?r ?to")}),
    ],
)

objects_by_type = {
    "robot": ["r1", "r2"],
    "location": ["roomA", "roomB"],
}
all_actions = move_op.instantiate(objects_by_type)
for a in all_actions:
    print(a.name)

In [ ]:
s0 = State(
    time=0,
    fluents={
        F("at r1 roomA"), F("free r1"),
        F("at r2 roomA"), F("free r2"),
    },
)

# kick off r1's move; r2 is still free so the clock stays at 0
a1 = get_action_by_name(get_next_actions(s0, all_actions), "move r1 roomA roomB")
s1, _ = transition(s0, a1)[0]
print("time:", s1.time)
print("r1 free?", F("free r1") in s1.fluents, "  r2 free?", F("free r2") in s1.fluents)
print("upcoming effects:", s1.upcoming_effects)

In [ ]:
# kick off r2's move; now nobody is free, clock fast-forwards to t=5
a2 = get_action_by_name(get_next_actions(s1, all_actions), "move r2 roomA roomB")
s2, _ = transition(s1, a2)[0]
print("time:", s2.time)
print("r1 at roomB?", F("at r1 roomB") in s2.fluents)
print("r2 at roomB?", F("at r2 roomB") in s2.fluents)
print("upcoming effects:", s2.upcoming_effects)

## Probabilistic Transitions

In [ ]:
search_cup = Action(
    name="search r1 roomA cup",
    preconditions={F("at r1 roomA"), F("free r1"), ~F("found cup")},
    effects=[
        GroundedEffect(0.0, {~F("free r1")}),
        GroundedEffect(
            3.0,
            resulting_fluents={F("free r1"), F("searched r1 roomA cup")},
            prob_effects=[
                # 80% branch: found the cup
                (0.8, [GroundedEffect(0.0, {F("found cup"), F("at cup r1")})]),
                # 20% branch: nothing extra happens
                (0.2, []),
            ],
        ),
    ],
)
print(search_cup)

In [ ]:
s0 = State(time=0, fluents={F("at r1 roomA"), F("free r1")})
outcomes = transition(s0, search_cup)
print(f"got {len(outcomes)} outcomes:")
for i, (s_out, p) in enumerate(outcomes):
    found = F("found cup") in s_out.fluents
    print(f"  branch {i}: prob={p:.2f}  time={s_out.time}  found cup? {found}")